# SMO — Demo del PMV

Recorre el flujo del laboratorio paso a paso:

**Datos curados → Base de datos → Objetos (nodos) → Red → Sub-red de trabajo → PageRank → Validación con Ashby**

Cada celda usa un módulo de `src/smo/`. Las decisiones abiertas están en `config/config.yaml`.

In [ ]:
import sys
from pathlib import Path
from copy import deepcopy
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

import pandas as pd
from smo.config import cargar_config
from smo.repositorio import crear_bd, RepositorioAleaciones
from smo.nodos import construir_nodos
from smo.tolerancias import calcular_tolerancias
from smo.redes import red_primaria, red_primigenia, red_completa, resumen
from smo.seleccion import seleccionar
from smo.validacion import indice_ashby, comparar

cfg = cargar_config()
cfg["pagerank"], cfg["tolerancias"], cfg["primigenia"]

## 1. Base de datos (Excel curado → SQLite)

In [ ]:
print(crear_bd(cfg["datos"]["excel"], cfg["datos"]["bd"]))
repo = RepositorioAleaciones(cfg["datos"]["bd"])

## 2. Nodos: un material = una fila de atributos

Granularidad `aleacion_estado`: un nodo por aleación + temple (decisión D1). Se descartan los nodos con atributos faltantes y los datos dudosos (D6).

In [ ]:
nodos, atributos = construir_nodos(repo, "aleacion_estado", list(cfg["atributos"]), excluir_dudosos=True)
print(len(nodos), "nodos;", nodos.attrs)
nodos.head(10)

## 3. Vecindades y redes

- **Red primaria**: una arista por atributo cuando las vecindades de dos nodos se sobrelapan, orientada hacia el valor creciente.
- **Red primigenia**: material → elemento aleante.

In [ ]:
tol = calcular_tolerancias(nodos, cfg["atributos"], atributos, "rango", 0.10)
primaria = red_primaria(nodos, tol, atributos)
primigenia = red_primigenia(nodos, repo.composicion(), 0.1, "hacia_elemento")
G = red_completa(primaria, primigenia)
resumen(G)

## 4. Selección: tirante liviano y resistente

Criterios: **maximizar Rp0,2** y **minimizar ρ**. Se arma la sub-red con esas aristas (más las de composición), se invierten las de ρ y se corre PageRank.

In [ ]:
caso = cfg["casos"]["tirante_liviano_resistente"]
ranking, H = seleccionar(G, caso["criterios"], alpha=0.85)
comp, metricas = comparar(ranking, indice_ashby(nodos, caso["indice_ashby"]))
print(metricas)
comp[["pagerank", "puesto", "indice_ashby", "puesto_ashby"]].head(10)

## 5. Figura: la sub-red sobre la carta de Ashby (ρ vs Rp0,2)

In [ ]:
from IPython.display import Image
from smo import visualizacion as viz
nombres = {a: d["nombre"] for a, d in cfg["atributos"].items()}
viz.red_sobre_carta(H, comp, nodos, ("densidad_kg_m3", "Rp02_MPa"), nombres, caso["descripcion"],
                    RAIZ / "outputs" / "demo_red.png")
Image(str(RAIZ / "outputs" / "demo_red.png"))

## 6. Cambiar una decisión abierta

Ejemplo: tolerancias **porcentuales** (±% del valor de cada nodo) en lugar de normalizadas por rango.

In [ ]:
tol_pct = calcular_tolerancias(nodos, cfg["atributos"], atributos, "porcentual")
G_pct = red_completa(red_primaria(nodos, tol_pct, atributos), primigenia)
rk_pct, _ = seleccionar(G_pct, caso["criterios"], alpha=0.85)
print("porcentual:", comparar(rk_pct, indice_ashby(nodos, caso["indice_ashby"]))[1])

## 7. Sensibilidad al ancho de vecindad (generada por `python ejecutar.py`)

In [ ]:
sens = pd.read_csv(RAIZ / "outputs" / "sensibilidad.csv")
display(sens.pivot_table(index="fraccion_rango", columns=["incluir_composicion", "caso"], values="spearman"))
Image(str(RAIZ / "outputs" / "sensibilidad.png"))